In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip
from qutip.measurement import measurement_statistics_observable

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track

# Linalg libraries
import numpy as np
import h5py as hf

# Helper libraries
from dataclasses import dataclass
import time

In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
background = np.genfromtxt("data/background.csv", delimiter=",", skip_header=1)[0:200, 1:]
signal = np.genfromtxt("data/signal.csv", delimiter=",", skip_header=1)[0:200, 1:]

In [ ]:
background = (background - background.mean(axis=0)) / background.std(axis=0)
signal = (signal - signal.mean(axis=0)) / signal.std(axis=0)

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length ** 2
    initial_state = []
    for i in range(number_of_spins):
        initial_state.append(
            basis(2, np.random.randint(low=0, high=2, size=1)[0])
        )

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]
    length = int(np.sqrt(N))

    # Magnetic field terms
    H = 0
    # for i in range(N):        
    H -= driving_field[t][0] * sz_list[0]
    H -= driving_field[t][1] * sz_list[-1]

    lattice_sites = []
    for x in range(length):
        for y in range(length):
            lattice_sites.append([x, y])

    lattice_sites = np.array(lattice_sites)
    neighbours = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]]) # right, left, top, bottom
    box_l = np.array([length, length])

    # Interaction terms
    spin_coupling_term = 0
    for n, site in enumerate(lattice_sites):

        neighbours = site[None, :] + neighbours
        neighbours_folded = neighbours - np.floor(neighbours / box_l[None, :]) * box_l[None, :]
        neighbour_indices = neighbours_folded[:, 0] + length * neighbours_folded[:, 1]

        for neighbour in neighbour_indices:
            spin_coupling_term += -0.5 * Jx[n] * sx_list[n] * sx_list[int(neighbour)]
            spin_coupling_term += -0.5 * Jy[n] * sy_list[n] * sy_list[int(neighbour)]
            spin_coupling_term += -0.5 * Jz[n] * sz_list[n] * sz_list[int(neighbour)]

    return H + 0.5 * spin_coupling_term  # account for double counting

In [ ]:
simulation_parameters = SimulationParameters(
    length=3,
    coupling=2.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": background
}

In [ ]:
# Observables for state description.
state_dimension = 20

measurements = []

for _ in range(state_dimension):
    seed = np.random.randint(641)
    measurements.append(
        qutip.rand_herm(
            4,
            0.5, 
            dims=[[2, 2], [2, 2]]
        )
    )

In [ ]:
signal_length = background.shape[0]
repeats = 5  # Number of times to measure a single operator

# Equilibration run
hamiltonian = compute_hamiltonian(0, args)
times = np.linspace(0, 1, 100)
new_state = mesolve(hamiltonian, simulation_state.quantum_state, times, [], [], args).states[-1]

observations = np.zeros((signal_length, state_dimension))
for t in track(range(signal_length), description="Running Simulation"):
    hamiltonian = compute_hamiltonian(t, args)
    for k, operator in enumerate(measurements):
        new_state = mesolve(hamiltonian, new_state, times, [], [], args).states[-1]
        measure_state = new_state.ptrace((1, 4))
        observations[t][k] = qutip.expect(measure_state * measure_state.dag(), operator)

np.save("background_observables.npy", observations)

In [ ]:
simulation_parameters = SimulationParameters(
    length=3,
    coupling=2.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": signal
}

signal_length = background.shape[0]
repeats = 5  # Number of times to measure a single operator

# Equilibration run
hamiltonian = compute_hamiltonian(0, args)
times = np.linspace(0, 1, 10)
new_state = mesolve(hamiltonian, simulation_state.quantum_state, times, [], [], args).states[-1]

observations = np.zeros((signal_length, state_dimension))
for t in track(range(signal_length), description="Running Simulation"):
    hamiltonian = compute_hamiltonian(t, args)
    for k, operator in enumerate(measurements):
        for j in range(repeats):
            new_state = mesolve(hamiltonian, new_state, times, [], [], args).states[-1]

            measure_state = new_state.ptrace((1, 4))

            eigs, states, probs = measurement_statistics_observable(measure_state, operator)

            i = np.random.choice(range(len(eigs)), p=np.real(probs) / np.real(probs).sum())

            
            value = eigs[i]
            observations[t][k][j] = value

np.save("signal_observables.npy", observations)

# Fit Model

In [ ]:
signal_ds = np.load("signal_observables.npy", allow_pickle=True)
background_ds = np.load("background_observables.npy", allow_pickle=True)

In [ ]:
def diagonal_matmul(a, b):
    return torch.diagonal(torch.matmul(a, b), 0)

In [ ]:
batched_matmul = torch.vmap(diagonal_matmul, in_dims=(0, None))

class ObservableEmbedding(nn.Module):
    """
    Custom embedding module for observables.

    Here we are essentially training a mean predictor.
    I do so by doing matmul with a trainable matrix and 
    taking only the diagonal.
    """
    def __init__(self, dimension: int, samples: int):
        """
        Constructor for the embedding module.
        """
        super().__init__()
        self.embedding = nn.Parameter(
            torch.rand(size=(samples, dimension))
            )

    def forward(self, x):
        """
        Forward pass through the embedding.
        """
        return batched_matmul(x, self.embedding)


class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, samples: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        samples : int
                Number of samples to take o each observable.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()


        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, 50),
            nn.ReLU(),
            nn.Linear(50, output_dimension),
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        function_data: np.ndarray,
        prediction_length: int
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data).to(device)
        self.function_data = torch.Tensor(function_data).to(device)
        
        self.norm_factor = 1 # max(abs(function_data.flatten()))
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx] / self.norm_factor
        
        return state, target

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: Avg loss: {test_loss:>8f} \n")
    
    return test_loss

In [ ]:
def train_model(input_data, target_data, model = None):
    """
    Train a model on the current data.
    """
    dataset = ReservoirDataset(
        state_data=input_data,
        function_data=target_data,
        prediction_length=1
    )
    
    if model is None:
        model = NeuralNetwork(
            state_dimension=20,
            samples=5,
            output_dimension=2
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Create the loader
    loader = DataLoader(dataset, batch_size=50, shuffle=False)
    
    # Train the network
    epochs = 20000
    train_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)

    train_losses = [item.cpu().detach().numpy() for item in train_losses]
    
    return train_losses, model

## Train the model

In [ ]:
background_ds.shape

In [ ]:
losses, model = train_model(background_ds, background, model=None)

In [ ]:
plt.plot(losses)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.show()

In [ ]:
background_predictions = model(torch.Tensor(background_ds).to(device)).cpu().detach().numpy()

In [ ]:
signal_predictions = model(torch.Tensor(signal_ds).to(device)).cpu().detach().numpy()

In [ ]:
plt.plot(signal[:, 0], signal_predictions[:, 0], 'r.', label="Predicted")
plt.plot(signal[:, 0], signal[:, 0], "k--", label="Actual")
plt.legend()
plt.savefig("signal-small-0.png")
plt.show()

In [ ]:
plt.plot(signal[:, 1], signal_predictions[:, 1], 'r.', label="Predicted")
plt.plot(signal[:, 1], signal[:, 1], "k--", label="Actual")
plt.legend()
plt.savefig("signal-small-1.png")
plt.show()

In [ ]:
plt.plot(background[:, 0], background[:, 0], 'k--', label="BG")
plt.plot(background[:, 0], background_predictions[:, 0], 'r.', label="Signal")
plt.legend()
plt.savefig("small-background-0.png")
plt.show()

In [ ]:
plt.plot(background[:, 1], background[:, 1], 'k--', label="BG")
plt.plot(background[:, 1], background_predictions[:, 1], 'r.', label="Signal")
plt.legend()
plt.savefig("small-background-1.png")
plt.show()